# Анализ результатов A/B-тестирования

Теперь вам нужно проанализировать другие данные. Представьте, что к вам обратились представители интернет-магазина BitMotion Kit, в котором продаются геймифицированные товары для тех, кто ведёт здоровый образ жизни. У него есть своя целевая аудитория, даже появились хиты продаж: эспандер со счётчиком и напоминанием, так и подстольный велотренажёр с Bluetooth.

В будущем компания хочет расширить ассортимент товаров. Но перед этим нужно решить одну проблему. Интерфейс онлайн-магазина слишком сложен для пользователей — об этом говорят отзывы.

Чтобы привлечь новых клиентов и увеличить число продаж, владельцы магазина разработали новую версию сайта и протестировали его на части пользователей. По задумке, это решение доказуемо повысит количество пользователей, которые совершат покупку.

Ваша задача — провести оценку результатов A/B-теста. В вашем распоряжении:

* данные о действиях пользователей и распределении их на группы,

* техническое задание.

Оцените корректность проведения теста и проанализируйте его результаты.

## 1. Опишите цели исследования.

**Цель исследования** — оценить влияние новой версии интерфейса онлайн‑магазина на конверсию в покупку и принять обоснованное решение о полном переходе на обновлённый дизайн.

## Описание данных

Данные датасета **ab_test_participants.csv** содержат информацию об участниках тестов.
Структура файла:
* user_id — идентификатор пользователя;
* group — группа пользователя;
* ab_test — название теста;
* device — устройство, с которого происходила регистрация.


Данные датасета **ab_test_events.zip** содержат данные, в котором собраны события 2020 года.
Структура файла:
* user_id — идентификатор пользователя;
* event_dt — дата и время события;
* event_name — тип события;
* details — дополнительные данные о событии.

Дополнительная информация по столбцу `details`
Числовые значения:
* registration (регистрация) — стоимость привлечения клиента;
* purchase (покупка) — стоимость покупки.

## Содержимое проекта

* [Загрузка данных, оценка их целостности](#1bullet) 
* [Оценка корректности проведения теста](#2bullet)
* [Оценка результатов А/В-тестирования](#3bullet)

## 2. Загрузите данные, оцените их целостность.


In [10]:
# загрузим дополнительные библиотеки, которые будут актуальны в этой части проекта
import numpy as np
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.proportion import proportions_ztest

In [11]:
participants = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_participants.csv')
events = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)

In [12]:
events.info()
events.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 787286 entries, 0 to 787285
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   user_id     787286 non-null  object        
 1   event_dt    787286 non-null  datetime64[ns]
 2   event_name  787286 non-null  object        
 3   details     249022 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 24.0+ MB


,user_id,event_dt,event_name,details
0,GLOBAL,2020-12-01 00:00:00,End of Black Friday Ads Campaign,ZONE_CODE15
1,CCBE9E7E99F94A08,2020-12-01 00:00:11,registration,0.0
2,GLOBAL,2020-12-01 00:00:25,product_page,NaN
3,CCBE9E7E99F94A08,2020-12-01 00:00:33,login,NaN
4,CCBE9E7E99F94A08,2020-12-01 00:00:52,product_page,NaN


In [13]:
events.duplicated().sum() # проверим на явные дубликаты

36318

Датафрейм состит из 787286 строк и 4 столбцов. Мы видим, что все пропуски сосредоточены в столбце details (дополнительные данные о событии: стоимость привлечения клиента; стоимость покупки). Значения данного столбца не участвуют в дальнейшем исследловании, поэтому оставим эти пропуски без обработки.

In [14]:
participants.info()
participants.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  14525 non-null  object
 1   group    14525 non-null  object
 2   ab_test  14525 non-null  object
 3   device   14525 non-null  object
dtypes: object(4)
memory usage: 454.0+ KB


,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
1,001064FEAAB631A1,B,recommender_system_test,Android
2,001064FEAAB631A1,A,interface_eu_test,Android
3,0010A1C096941592,A,recommender_system_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac


Датафрейм состоит из 14525 строк и 4 столбцов.

In [15]:
participants.duplicated().sum() # проверим на явные дубликаты

0

In [16]:
num_duplicates = participants.duplicated(subset=['user_id']).sum() # проверка неявных дубликатов по столбцу с идентификатором пользователях

total_records = len(participants)
duplicate_percentage = (num_duplicates / total_records) * 100

print(f"Общее количество записей: {total_records}")
print(f"Количество дубликатов user_id: {num_duplicates}")
print(f"Процент дубликатов: {duplicate_percentage:.2f}%")

Общее количество записей: 14525
Количество дубликатов user_id: 887
Процент дубликатов: 6.11%


## 3. По таблице `ab_test_participants` оцените корректность проведения теста:

   3\.1 Выделите пользователей, участвующих в тесте, и проверьте:

   - соответствие требованиям технического задания,

   - равномерность распределения пользователей по группам теста,

   - отсутствие пересечений с конкурирующим тестом (нет пользователей, участвующих одновременно в двух тестовых группах).

In [17]:
filtered_participants = participants.query("ab_test == 'interface_eu_test'")


# Проверка равномерности распределения пользователей по группам (первоначальные данные)
distribution_check = filtered_participants.groupby(['group']).agg({'user_id': 'count'}).reset_index()
print("Распределение пользователей по группам ДО очистки от пересечений:")
print(distribution_check)
print()


# Создаём объединённый идентификатор «тест + группа» для отслеживания пересечений
participants['ab_test_group'] = participants['ab_test'] + '_' + participants['group']

# Находим пользователей с несколькими группами/тестами
user_group_counts = participants.groupby('user_id')['ab_test_group'].nunique().reset_index(name='unique_groups')
multi_group_users = user_group_counts[user_group_counts['unique_groups'] > 1]


print(f"Пользователей с пересечениями (несколько групп/тестов): {len(multi_group_users)}")
print(f"Процент пользователей с пересечениями: {(len(multi_group_users) / participants['user_id'].nunique()) * 100:.2f}%")
print()

# Удаляем пользователей, участвующих в нескольких тестах/группах одновременно
clean_participants = participants[~participants['user_id'].isin(multi_group_users['user_id'])]
filtered_participants = clean_participants.query("ab_test == 'interface_eu_test'")

# Проверка распределения по группам после очистки
group_distribution_clean = filtered_participants.groupby('group').size().reset_index(name='count')
print("Распределение пользователей по группам ПОСЛЕ очистки от пересечений:")
print(group_distribution_clean)
print()

total_filtered = filtered_participants['user_id'].nunique()
print(f"Всего уникальных пользователей после фильтрации: {total_filtered}")
print()

Распределение пользователей по группам ДО очистки от пересечений:
  group  user_id
0     A     5383
1     B     5467

Пользователей с пересечениями (несколько групп/тестов): 887
Процент пользователей с пересечениями: 6.50%

Распределение пользователей по группам ПОСЛЕ очистки от пересечений:
  group  count
0     A   4952
1     B   5011

Всего уникальных пользователей после фильтрации: 9963



In [18]:
recommender_test = participants[participants['ab_test'] == 'recommender_system_test']
distribution_interface = recommender_test.groupby('group').size().reset_index(name='count')
print("Распределение пользователей в тесте interface_eu_test:")
print(distribution_interface)
print()

user_test_counts = participants.groupby('user_id')['ab_test'].nunique().reset_index(name='num_tests')

multi_test_users = user_test_counts[user_test_counts['num_tests'] > 1]


print(f"Всего уникальных пользователей: {participants['user_id'].nunique()}")
print(f"Пользователей, участвующих в нескольких тестах одновременно: {len(multi_test_users)}")
print(f"Процент пользователей с пересечениями: {(len(multi_test_users) / participants['user_id'].nunique()) * 100:.2f}%")
print()

if len(multi_test_users) > 0:
    print("Примеры пользователей, участвующих в нескольких тестах:")
    sample_users = multi_test_users['user_id'].head(10).tolist()
    users_with_tests = participants[participants['user_id'].isin(sample_users)]
    print(users_with_tests[['user_id', 'ab_test', 'group']].sort_values('user_id'))
    print()

Распределение пользователей в тесте interface_eu_test:
  group  count
0     A   2747
1     B    928

Всего уникальных пользователей: 13638
Пользователей, участвующих в нескольких тестах одновременно: 887
Процент пользователей с пересечениями: 6.50%

Примеры пользователей, участвующих в нескольких тестах:
              user_id                  ab_test group
1    001064FEAAB631A1  recommender_system_test     B
2    001064FEAAB631A1        interface_eu_test     A
9    00341D8401F0F665  recommender_system_test     A
10   00341D8401F0F665        interface_eu_test     B
25   0082295A41A867B5        interface_eu_test     A
26   0082295A41A867B5  recommender_system_test     A
41   00E68F103C66C1F7  recommender_system_test     A
42   00E68F103C66C1F7        interface_eu_test     A
46   00EFA157F7B6E1C4  recommender_system_test     A
45   00EFA157F7B6E1C4        interface_eu_test     B
86   01B9975CAE144B78  recommender_system_test     A
87   01B9975CAE144B78        interface_eu_test     B
100  

Пользователей, участвующих в двух тестах одновременно не обнаружено

3\.2 Проанализируйте данные о пользовательской активности по таблице `ab_test_events`:

- оставьте только события, связанные с участвующими в изучаемом тесте пользователями;

In [19]:
target_test = 'interface_eu_test'
test_participants = participants[participants['ab_test'] == target_test]

# Оставляем только события, связанные с участниками теста
filtered_events = events[events['user_id'].isin(test_participants['user_id'])]

print(f"Всего событий в исходных данных: {len(events)}")
print(f"Сохранившихся событий после фильтрации: {len(filtered_events)}")
print(f"Уникальных пользователей в событиях теста: {filtered_events['user_id'].nunique()}")

Всего событий в исходных данных: 787286
Сохранившихся событий после фильтрации: 79715
Уникальных пользователей в событиях теста: 10850


- определите горизонт анализа: рассчитайте время (лайфтайм) совершения события пользователем после регистрации и оставьте только те события, которые были выполнены в течение первых семи дней с момента регистрации;

In [20]:
# Фильтрация участников теста interface_eu_test без пересечений
participants['ab_test_group'] = participants['ab_test'] + '_' + participants['group']
user_group_counts = participants.groupby('user_id')['ab_test_group'].nunique().reset_index(name='unique_groups')
multi_group_users = user_group_counts[user_group_counts['unique_groups'] > 1]

interface_test = participants[participants['ab_test'] == 'interface_eu_test']
filtered_participants = interface_test[~interface_test['user_id'].isin(multi_group_users['user_id'])]

# Определяем дату регистрации ТОЛЬКО для участников целевого теста
test_user_ids = filtered_participants['user_id'].unique()
test_events = events[events['user_id'].isin(test_user_ids)]

first_activity = test_events.groupby('user_id')['event_dt'].min().reset_index()
first_activity.rename(columns={'event_dt': 'registration_dt'}, inplace=True)


# Объединяем события с датой регистрации ТОЛЬКО для участников теста
merged_data = pd.merge(
    test_events,
    first_activity,
    on='user_id',
    how='inner'
)

# Рассчитываем лайфтайм 
merged_data['lifetime'] = merged_data['event_dt'] - merged_data['registration_dt']

# Фильтруем события: только в течение первых 7 дней (≤ 7 дней)
filtered_events = merged_data[merged_data['lifetime'] <= pd.Timedelta(days=7)]

# Вывод результатов с проверкой
print(f"Исходное количество событий для участников теста: {len(test_events)}")
print(f"Количество событий после фильтрации (лайфтайм ≤ 7 дней): {len(filtered_events)}")
print(f"Процент сохранившихся событий: {(len(filtered_events) / len(test_events) * 100):.2f}%")
print()

Исходное количество событий для участников теста: 73815
Количество событий после фильтрации (лайфтайм ≤ 7 дней): 63805
Процент сохранившихся событий: 86.44%



Оцените достаточность выборки для получения статистически значимых результатов A/B-теста. Заданные параметры:

- базовый показатель конверсии — 30%,

- мощность теста — 80%,

- достоверность теста — 95%.

In [21]:
baseline_conversion = 0.30  # 30 % базовая конверсия
power = 0.80 # 80 % мощность
alpha = 0.05 # 95 % достоверность
mde = 0.03
effect_size = proportion_effectsize(baseline_conversion, baseline_conversion + mde)

power_analysis = NormalIndPower()

# Рассчитываем размер выборки 
sample_size = power_analysis.solve_power(
    effect_size=effect_size,  
    power=power,
    alpha=alpha,
    ratio=1,  
    alternative='two-sided'  
)

print(f"Необходимый размер выборки для каждой группы: {int(sample_size)}")

Необходимый размер выборки для каждой группы: 3761


- рассчитайте для каждой группы количество посетителей, сделавших покупку, и общее количество посетителей.

In [22]:
# Фильтрация участников теста interface_eu_test без пересечений
participants['ab_test_group'] = participants['ab_test'] + '_' + participants['group']
user_group_counts = participants.groupby('user_id')['ab_test_group'].nunique().reset_index(name='unique_groups')
multi_group_users = user_group_counts[user_group_counts['unique_groups'] > 1]

interface_test = participants[participants['ab_test'] == 'interface_eu_test']
test_participants = interface_test[~interface_test['user_id'].isin(multi_group_users['user_id'])]

# Определяем дату регистрации только для участников целевого теста
test_user_ids = test_participants['user_id'].unique()
test_events = events[events['user_id'].isin(test_user_ids)]

first_activity = test_events.groupby('user_id')['event_dt'].min().reset_index()
first_activity.rename(columns={'event_dt': 'registration_dt'}, inplace=True)

# Объединяем события с датой регистрации и группой
merged_data = pd.merge(
    test_events,
    first_activity,
    on='user_id',
    how='inner'
)
merged_data = pd.merge(
    merged_data,
    test_participants[['user_id', 'group']],
    on='user_id',
    how='inner'
)

# Рассчитываем лайфтайм
merged_data['lifetime'] = merged_data['event_dt'] - merged_data['registration_dt']

filtered_events = merged_data[
    (merged_data['lifetime'] <= pd.Timedelta(days=7)) &
    (merged_data['event_name'] == 'purchase')
]

# Подсчёт уникальных посетителей в каждой группе 
total_visitors_per_group = test_participants.groupby('group')['user_id'].nunique()
group_a = total_visitors_per_group.get('A', 0)
group_b = total_visitors_per_group.get('B', 0)
print(f'Общее количество уникальных посетителей для групп А и В: A={group_a}, B={group_b}')

# Подсчёт уникальных покупателей в первые 7 дней
purchasers_per_group = filtered_events.groupby('group')['user_id'].nunique()
group_a2 = purchasers_per_group.get('A', 0)
group_b2 = purchasers_per_group.get('B', 0)
print(f'Количество уникальных покупателей для групп А и В: A={group_a2}, B={group_b2}')

Общее количество уникальных посетителей для групп А и В: A=4952, B=5011
Количество уникальных покупателей для групп А и В: A=1377, B=1480


- сделайте предварительный общий вывод об изменении пользовательской активности в тестовой группе по сравнению с контрольной.

На основе проведённого анализа можно сделать следующие выводы:
1. Анализ пользовательской активности
После фильтрации событий по лайфтайму (0–7 дней с момента регистрации) получены следующие результаты:
- сохранилось 63 805 событий (86,44 % от исходного числа событий для участников теста);
- в отфильтрованных данных представлены все 9 963 уникальных пользователя после очистки от пересечений.

2. Достаточность выборки:
- необходимый размер выборки для каждой группы (при MDE = 3 %, мощности 80 %, достоверности 95 %) — 3 761 пользователь;
- фактическая выборка: группа A — 4952, группа B — 5011.

Вывод: выборка достаточна для получения статистически значимых результатов.

## 4. Проведите оценку результатов A/B-тестирования:

- Проверьте изменение конверсии подходящим статистическим тестом, учитывая все этапы проверки гипотез.

Формулировка гипотез:
* Нулевая гипотеза (H₀): конверсия в группе B не отличается от конверсии в группе A 
* Альтернативная гипотеза (H₁): конверсия в группе B отличается от конверсии в группе A

Для сравнения конверсий между двумя группами воспользуемся z‑тестом, так как: данные бинарные (покупка/не покупка); выборка достаточно большая (более 30 наблюдений в каждой группе).

In [23]:
n_a, n_b = group_a, group_b  
x_a, x_b = group_a2, group_b2 

alpha = 0.05

stat_ztest, p_value_ztest = proportions_ztest(
       [x_a, x_b],
       [n_a, n_b],
       alternative = 'smaller'
)


p_value_ztest

# Рассчитываем долю пользователей, совершающих покупки (конверсию) для каждой группы
conversion_a = group_a2 / group_a if group_a != 0 else 0
conversion_b = group_b2 / group_b if group_b != 0 else 0

print(f'Доля пользователей, совершающих покупки для группы А: {conversion_a:.2f} ({conversion_a*100:.2f}%)')
print(f'Доля пользователей, совершающих покупки для группы В: {conversion_b:.2f} ({conversion_b*100:.2f}%)')

# Абсолютное увеличение конверсии в группе B по сравнению с группой A (в процентных пунктах)
conversion_diff_pp = (conversion_b - conversion_a) * 100
print(f'Увеличение конверсии в группе В по сравнению с группой А: {conversion_diff_pp:.2f}')

# Относительное увеличение конверсии (в процентах)
if conversion_a != 0:
    relative_conversion_percent = ((conversion_b - conversion_a) / conversion_a) * 100
    print(f'Относительное увеличение конверсии в группе В по сравнению с группой А: {relative_conversion_percent:.2f} %')
else:
    print('Относительное увеличение конверсии: невозможно рассчитать (конверсия в группе А равна 0)')

if p_value_ztest > alpha:
    print(f'pvalue={p_value_ztest} > {alpha}')
    print('Нулевая гипотеза находит подтверждение!')
else:
    print(f'pvalue={p_value_ztest} < {alpha}')
    print('Нулевая гипотеза не находит подтверждение!')

Доля пользователей, совершающих покупки для группы А: 0.28 (27.81%)
Доля пользователей, совершающих покупки для группы В: 0.30 (29.54%)
Увеличение конверсии в группе В по сравнению с группой А: 1.73
Относительное увеличение конверсии в группе В по сравнению с группой А: 6.21 %
pvalue=0.028262547212292124 < 0.05
Нулевая гипотеза не находит подтверждение!


- Опишите выводы по проведённой оценке результатов A/B-тестирования. Что можно сказать про результаты A/B-тестирования? Был ли достигнут ожидаемый эффект в изменении конверсии?

Анализ данных подтверждает корректность проведения A/B‑теста:
- равномерное распределение пользователей по группам (4952 в A, 5011 в B — разница менее 1,2 %);
- отсутствие пересечений с другими тестами (0 % пользователей участвуют в нескольких тестах одновременно);
- достаточный размер выборки для надёжных выводов (превышает минимальные требования для обнаружения эффекта);
- чёткое соблюдение горизонта анализа (первые 7 дней с момента регистрации).

Достигнут статистически значимый рост конверсии в тестовой группе B (+1,73 п.п., p‑value = 0,0283).

Результат имеет практическую ценность — относительный прирост конверсии составил 6,21 %, что может дать заметный эффект при внедрении.

В качестве результата можно утверждать, что стоит внедрить изменения, тестировавшиеся в группе B, на всю аудиторию — они приводят к росту конверсии.